# Laboratório: RAG

**Dupla:**

- Nome 1: José Cássio
- Nome 2: Felipe Baraúna

Neste laboratório deverá ser montado um RAG completo sobre documentos escolhidos pela dupla. Pode ser um regulamento, um manual, uma coleção de artigos, a documentação de uma biblioteca ou qualquer conjunto de textos conhecido o bastante para julgar se uma resposta está certa. As etapas são carregar os modelos, carregar os documentos, cortar em trechos, indexar, recuperar, gerar a resposta e avaliar o resultado. Cada seção descreve o que se espera dela e traz um espaço vazio para o código, e podem ser criadas quantas células forem precisas.

A avaliação depende de um benchmark elaborado pela dupla: um conjunto de perguntas sobre os documentos, cada uma com a resposta esperada e o lugar do documento onde ela está. O benchmark deve ser escrito cedo, de preferência antes de testar o sistema, para que as perguntas não sejam escolhidas a partir do que já funciona. Ele deve incluir perguntas fáceis, perguntas cuja resposta está espalhada em mais de um trecho e pelo menos uma que os documentos não respondem.

In [1]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langchain-community langchain-chroma langgraph pdfplumber beautifulsoup4
# !pip install -q langchain-groq langchain-google-genai
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [2]:
import json
import os
from pathlib import Path

import pandas as pd
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import (
    DirectoryLoader,
    PDFPlumberLoader,
    TextLoader,
    WebBaseLoader,
)
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

/tmp/ipykernel_224736/1181073127.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
import hashlib
import re

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from fastembed import TextEmbedding
from fastembed.common.model_description import ModelSource, PoolingType
from langchain_core.embeddings import Embeddings

load_dotenv("../.env")
pd.set_option("display.max_colwidth", 120)

### Benchmark

O benchmark foi escrito antes de qualquer teste, a partir da leitura dos documentos, e por isso fica no começo do notebook: as seções de retrieval e de geração já usam as perguntas dele. Ele é mostrado e explicado na seção Avaliação.

In [4]:
def p(documento, pagina):
    return {"documento": documento, "pagina": pagina}


def s(documento, secao):
    return {"documento": documento, "secao": secao}


BENCHMARK = [
    {"id": 1, "tipo": "fácil", "pergunta": "Qual é a faixa de tensão de alimentação do sensor DHT22?",
     "resposta": "3,3 a 5,5 V DC (a tabela elétrica dá mínimo 3,3 V, típico 5 V e máximo 6 V).",
     "fontes": [[p("dht22", 1), p("dht22", 2), p("dht22", 5)]]},
    {"id": 2, "tipo": "fácil", "pergunta": "De quanto em quanto tempo posso fazer uma leitura do DHT22?",
     "resposta": "O período de coleta é de 2 segundos: as leituras devem ter pelo menos 2 s de intervalo.",
     "fontes": [[p("dht22", 3), p("dht22", 5)]]},
    {"id": 3, "tipo": "fácil", "pergunta": "Qual a distância mínima e máxima que o HC-SR04 mede, e com que precisão?",
     "resposta": "De 2 cm a 400 cm (4 m), com precisão de até 3 mm.",
     "fontes": [[p("hc-sr04", 1)]]},
    {"id": 4, "tipo": "fácil", "pergunta": "Qual a tensão de trabalho e a corrente do HC-SR04?",
     "resposta": "5 V DC e 15 mA.",
     "fontes": [[p("hc-sr04", 1)]]},
    {"id": 5, "tipo": "fácil", "pergunta": "Como converto a largura do pulso do Echo do HC-SR04 em centímetros?",
     "resposta": "Distância em cm = tempo em µs / 58 (ou tempo do nível alto x 340 m/s / 2).",
     "fontes": [[p("hc-sr04", 2), p("hc-sr04", 1)]]},
    {"id": 6, "tipo": "fácil", "pergunta": "Quanta corrente um pino GPIO do ESP32 consegue fornecer?",
     "resposta": "Cerca de 40 mA nos domínios VDD3P3_CPU e VDD3P3_RTC (20 mA no VDD_SDIO), caindo para cerca de "
                 "29 mA por pino quando muitos pinos fornecem corrente ao mesmo tempo.",
     "fontes": [[p("esp32-datasheet", 52)]]},
    {"id": 7, "tipo": "fácil", "pergunta": "Quanto o ESP32 consome em deep sleep?",
     "resposta": "10 µA com só o RTC timer e a memória RTC; 150 µA com o coprocessador ULP ligado; "
                 "100 µA a 1% de ciclo no padrão monitorado pelo ULP.",
     "fontes": [[p("esp32-datasheet", 30)]]},
    {"id": 8, "tipo": "fácil", "pergunta": "Quais são os strapping pins do ESP32 e o nível padrão de cada um?",
     "resposta": "GPIO0 (pull-up, 1), GPIO2 (pull-down, 0), MTDI (pull-down, 0), MTDO (pull-up, 1) e GPIO5 (pull-up, 1).",
     "fontes": [[p("esp32-datasheet", 22)]]},
    {"id": 9, "tipo": "fácil", "pergunta": "Quantos GPIOs programáveis o ESP32 tem e quantos deles são só de entrada?",
     "resposta": "34 GPIOs programáveis, dos quais 6 são só de entrada (e 5 são strapping pins).",
     "fontes": [[p("esp32-datasheet", 4)]]},
    {"id": 10, "tipo": "fácil", "pergunta": "Qual a tensão máxima absoluta que posso aplicar na alimentação do ESP32?",
     "resposta": "3,6 V (a faixa absoluta é de -0,3 V a 3,6 V).",
     "fontes": [[p("esp32-datasheet", 51)]]},
    {"id": 11, "tipo": "fácil", "pergunta": "Qual a largura mínima de trilha e o espaçamento que a JLCPCB fabrica numa placa de 2 camadas com cobre de 1 oz?",
     "resposta": "0,10 mm / 0,10 mm (4 / 4 mil).",
     "fontes": [[s("jlcpcb", "Traces")]]},
    {"id": 12, "tipo": "espalhada", "pergunta": "Posso alimentar o DHT22 e o HC-SR04 com o 3,3 V do ESP32 e ligar os sinais direto num GPIO?",
     "resposta": "O DHT22 sim, ele aceita 3,3 a 5,5 V. O HC-SR04 trabalha em 5 V, e o Echo em 5 V passa do máximo de entrada "
                 "do GPIO do ESP32 (VDD + 0,3 V, cerca de 3,6 V), então precisa de um divisor de tensão.",
     "fontes": [[p("dht22", 1), p("dht22", 2), p("dht22", 5)], [p("hc-sr04", 1)],
                [p("esp32-datasheet", 52), p("esp32-datasheet", 51)]]},
    {"id": 13, "tipo": "espalhada", "pergunta": "Somando o ESP32 em deep sleep (só RTC timer e memória RTC) e o DHT22 em stand-by, quanto o circuito consome?",
     "resposta": "Cerca de 50 a 60 µA: 10 µA do ESP32 mais 40 a 50 µA do DHT22 em stand-by.",
     "fontes": [[p("esp32-datasheet", 30)], [p("dht22", 5)]]},
    {"id": 14, "tipo": "sem resposta", "pergunta": "Qual a faixa de temperatura de operação do HC-SR04?",
     "resposta": "Os documentos não informam.", "fontes": []},
]
print(len(BENCHMARK), "perguntas")

14 perguntas


## Modelos

Carregamento do modelo de chat que vai gerar as respostas e do modelo de embedding que vai representar os trechos e as perguntas. O modelo de embedding precisa ser o mesmo na indexação e na busca, e trocá-lo depois obriga a refazer o índice, porque o tamanho do vetor muda.

O modelo de chat vem da variável `MODELO` do `.env` (o nome do modelo usado nesta execução é impresso na célula abaixo). A chave fica só no `.env`, fora do zip.

O embedding é o `intfloat/multilingual-e5-small`, rodando local em ONNX pelo `fastembed`, o mesmo do nosso Assistente de Bancada. Escolhemos um modelo multilíngue porque as perguntas são em português e os documentos em inglês. O e5 exige os prefixos `passage: ` nos trechos e `query: ` nas perguntas, então os dois passam só por esta classe. Na primeira execução o modelo (cerca de 470 MB) é baixado para `modelos/`.

In [5]:
model = init_chat_model(os.environ.get("MODELO", "google_genai:gemini-3.5-flash-lite"), temperature=0)

E5 = "intfloat/multilingual-e5-small"


class EmbeddingsE5(Embeddings):
    def __init__(self, pasta="modelos"):
        if E5 not in {m["model"] for m in TextEmbedding.list_supported_models()}:
            TextEmbedding.add_custom_model(model=E5, pooling=PoolingType.MEAN, normalization=True,
                                           sources=ModelSource(hf=E5), dim=384, model_file="onnx/model.onnx")
        self.modelo = TextEmbedding(E5, cache_dir=pasta)

    def embed_documents(self, textos):
        return [v.tolist() for v in self.modelo.embed([f"passage: {t}" for t in textos], batch_size=32)]

    def embed_query(self, texto):
        return next(iter(self.modelo.embed([f"query: {texto}"]))).tolist()


embeddings = EmbeddingsE5()
print("modelo:", os.environ.get("MODELO"))
print(model.invoke("Responda só com a palavra ok.").text, "| dimensão do embedding:", len(embeddings.embed_query("teste")))

modelo: google_genai:gemini-3.5-flash-lite


/home/jcassio-dev/www/labs-unidade2/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Ok. | dimensão do embedding: 384


## Loading

Descrição breve dos documentos escolhidos, com a origem, o formato e o tipo de pergunta que eles respondem, seguida do carregamento de tudo como uma lista de `Document`, com o loader adequado ao formato.

Usamos quatro documentos que o nosso Assistente de Bancada já consulta, todos em inglês, na pasta `documentos/`:

| id | documento | origem | formato | responde |
|---|---|---|---|---|
| `esp32-datasheet` | ESP32 Series Datasheet v5.3 | Espressif | PDF, 78 páginas | pinos, boot, consumo, características elétricas |
| `dht22` | AM2302 (DHT22) | Aosong (cópia da Adafruit) | PDF, 5 páginas | alimentação, protocolo, tempos do sensor |
| `hc-sr04` | Ultrasonic Ranging Module HC-SR04 | Elecfreaks | PDF, 3 páginas | alimentação, alcance, cálculo da distância |
| `jlcpcb` | Rigid PCB Manufacturing Capabilities | JLCPCB | HTML | limites de fabricação de placas |

São perguntas de quem monta um circuito com ESP32: qual a tensão de um sensor, quanto corrente um pino aguenta, qual trilha o fabricante consegue fazer. Conhecemos os documentos o bastante para julgar as respostas.

Os PDFs são lidos com o `PDFPlumberLoader` (um `Document` por página). O HTML da JLCPCB é lido com o BeautifulSoup, um `Document` por seção (`h3`), com cada linha das tabelas no formato `coluna: valor`, para o trecho não perder o cabeçalho.

In [6]:
PASTA = Path("documentos")
TITULOS = {
    "esp32-datasheet": "ESP32 Series Datasheet",
    "dht22": "AM2302 (DHT22) Datasheet",
    "hc-sr04": "HC-SR04 Ultrasonic Ranging Module",
    "jlcpcb": "JLCPCB Rigid PCB Manufacturing Capabilities",
}

cru = PDFPlumberLoader(str(PASTA / "esp32-datasheet.pdf")).load()
print("Padrão do PDFPlumberLoader:\n", cru[3].page_content[:200])

paginas = []
for documento in ["esp32-datasheet", "dht22", "hc-sr04"]:
    paginas += PDFPlumberLoader(str(PASTA / f"{documento}.pdf"), text_kwargs={"x_tolerance": 1.5}).load()
print("\nCom x_tolerance=1.5:\n", paginas[3].page_content[:200])


def carregar_html(caminho):
    sopa = BeautifulSoup(caminho.read_text(), "html.parser")
    secoes, secao, linhas, cabecalho = [], None, [], []
    for el in sopa.find("main").find_all(["h2", "h3", "table", "p"]):
        if el.name in ("h2", "h3"):
            if secao and linhas:
                secoes.append((secao, linhas))
            secao, linhas = el.get_text(" ", strip=True), []
        elif el.name == "table":
            tabela = [[c.get_text(" ", strip=True) for c in tr.find_all(["th", "td"])] for tr in el.find_all("tr")]
            if len(tabela) == 1:
                cabecalho = tabela[0]
                continue
            for linha in tabela:
                linhas.append(" | ".join(f"{cabecalho[j]}: {c}" if j < len(cabecalho) else c
                                         for j, c in enumerate(linha) if c))
        elif el.find_parent("table") is None:
            texto = el.get_text(" ", strip=True)
            if texto:
                linhas.append(texto)
    if secao and linhas:
        secoes.append((secao, linhas))
    return [Document("\n".join(l), metadata={"source": str(caminho), "secao": s}) for s, l in secoes]


secoes_html = carregar_html(PASTA / "jlcpcb.html")
docs = paginas + secoes_html
print(f"\n{len(paginas)} páginas de PDF e {len(secoes_html)} seções de HTML")

Padrão do PDFPlumberLoader:
 – 2coresat240MHz: 1079.96CoreMark;4.50CoreMark/MHz
• 448KBROM
• 520KBSRAM
• 16KBSRAMinRTC
• QSPIsupportsmultipleflash/SRAMchips
ClocksandTimers
• Internal8MHzoscillatorwithcalibration
• InternalRCosci



Com x_tolerance=1.5:
 – 2 cores at 240 MHz: 1079.96 CoreMark; 4.50 CoreMark/MHz
• 448 KB ROM
• 520 KB SRAM
• 16 KB SRAM in RTC
• QSPI supports multiple flash/SRAM chips
Clocks and Timers
• Internal 8 MHz oscillator with ca

86 páginas de PDF e 7 seções de HTML


### Inspeção e metadados

O texto que sai do loader deve ser inspecionado antes de seguir, porque quebras de linha, cabeçalhos repetidos, tabelas e páginas vazias afetam todas as etapas seguintes. Os metadados também devem ser conferidos e completados com o que for útil para citar a fonte ou filtrar a busca depois, como o nome do arquivo, a página corrigida ou a seção.

In [7]:
print(docs[78].page_content[:400])
print("---")
print(paginas[51].page_content[-160:])

YYYYour specialist in innovating humidity & temperature sensors
SSSSttttaaaannnnddddaaaarrrrdddd AAAAMMMM2222333300002222////DDDDHHHHTTTT22222222 AAAAMMMM2222333300002222////DDDDHHHHTTTT22222222 wwwwiiiitttthhhh bbbbiiiigggg ccccaaaasssseeee aaaannnndddd wwwwiiiirrrreeeessss
DDDDiiiiggggiiiittttaaaallll rrrreeeellllaaaattttiiiivvvveeee hhhhuuuummmmiiiiddddiiiittttyyyy &&&& tttteeeemmmmppppeeeerrrr
---
02.11n, OFDM MCS7, POUT = +14 dBm — 180 — mA
Receive 802.11b/g/n — 95 ~ 100 — mA
Espressif Systems 52 ESP32 Series Datasheet v5.3
Submit Documentation Feedback



A inspeção mostrou três problemas:

- Com o padrão do `pdfplumber` (`x_tolerance=3`), o datasheet do ESP32 sai com as palavras grudadas ("2coresat240MHz"), o que estraga a busca por palavras e os embeddings. Com `x_tolerance=1.5` os espaços voltam.
- O PDF do DHT22 simula negrito desenhando o texto quatro vezes, e o texto sai como "PPPPoooowwwweeeerrrr". Isso aparece justamente nos títulos e em valores como "+-2%RH".
- Todas as páginas têm cabeçalho e rodapé repetidos (o rodapé da Espressif com número da página e o rodapé do DHT22 com nome e email do vendedor), que não respondem nada e aparecem em todos os trechos.

A limpeza abaixo junta as letras repetidas, tira os rodapés e as páginas vazias e completa os metadados: `documento` (id usado no filtro), `titulo` (para a citação), `pagina` começando em 1 (o loader conta a partir de 0) e `secao` no HTML.

In [8]:
RODAPES = [
    r"Espressif\s*Systems\s*\d+\s*ESP32\s*Series\s*Datasheet\s*v[\d.]+\s*Submit\s*Documentation\s*Feedback",
    r"Thomas\s*Liu\s*\(Sales\s*Manager\)\s*-\d+-\s*Email:\s*\S+",
    r"Your\s*specialist\s*in\s*innovating\s*humidity\s*&\s*temperature\s*sensors",
    r"Tech Support: \S+",
    r"www\.Elecfreaks\.com",
]


def sem_repeticoes(texto):
    texto = re.sub(r"(\S)\1{3}", r"\1", texto)
    return re.sub(r"(?<!\S)(\S+)(?:\s+\1){2,}(?!\S)", r"\1", texto)


def limpar(doc):
    documento = Path(doc.metadata["source"]).stem
    texto = sem_repeticoes(doc.page_content) if documento == "dht22" else doc.page_content
    for rodape in RODAPES:
        texto = re.sub(rodape, "", texto, flags=re.I)
    texto = texto.replace("(cid:1)", "-").strip()
    meta = {"documento": documento, "titulo": TITULOS[documento], "source": Path(doc.metadata["source"]).name}
    if "page" in doc.metadata:
        meta["pagina"] = doc.metadata["page"] + 1
    else:
        meta["secao"] = doc.metadata["secao"]
    return Document(texto, metadata=meta)


limpos = [limpar(d) for d in docs]
limpos = [d for d in limpos if len(d.page_content) > 50]
print(f"{len(limpos)} documentos depois da limpeza ({len(docs) - len(limpos)} páginas vazias removidas)\n")
dht = next(d for d in limpos if d.metadata["documento"] == "dht22" and d.metadata["pagina"] == 1)
print(dht.metadata, "\n", dht.page_content[:600])
pd.DataFrame([d.metadata | {"caracteres": len(d.page_content)} for d in limpos]).groupby("documento")["caracteres"].describe()

93 documentos depois da limpeza (0 páginas vazias removidas)

{'documento': 'dht22', 'titulo': 'AM2302 (DHT22) Datasheet', 'source': 'dht22.pdf', 'pagina': 1} 
 Standard AM2302/DHT22 AM2302/DHT22 with big case and wires
Digital relative humidity & temperature sensor AM2302/DHT22
1. Feature & Application:
*High precision *Outstanding long-term stability
*Capacitive type *Extra components not needed
*Full range temperature compensated *Long transmission distance, up to 100 meters
*Relative humidity and temperature measurement *Low power consumption
*Calibrated digital signal *4 pins packaged and fully interchangeable
2. Description:
AM2302 output calibrated digital signal. It applys exclusive digital-signal-collecting-technique and humidity
sensing tech


,count,mean,std,min,25%,50%,75%,max
documento,,,,,,,,
dht22,5.0,1367.600000,473.331596,800.0,931.0,1533.0,1741.0,1833.0
esp32-datasheet,78.0,1518.910256,687.544722,129.0,1077.5,1569.5,1881.0,4892.0
hc-sr04,3.0,684.000000,312.558794,391.0,519.5,648.0,830.5,1013.0
jlcpcb,7.0,2211.571429,1697.059404,119.0,1056.5,1816.0,3341.0,4751.0


## Chunking

Corte dos documentos em trechos. A escolha do cortador depende da estrutura do texto, e o `chunk_size` e o `chunk_overlap` dependem do tamanho das unidades de sentido do documento, como um artigo, um parágrafo ou uma seção. A escolha deve ser justificada com base no que a inspeção mostrou.

O texto que sai das páginas não tem parágrafos separados por linha em branco: cada linha é uma linha do PDF, e as tabelas viram uma linha por registro. Por isso usamos o `RecursiveCharacterTextSplitter`, que tenta cortar primeiro em `\n\n`, depois em `\n` e só então no meio da linha, o que preserva as linhas das tabelas inteiras.

A unidade de sentido aqui é uma tabela ou um bloco de uma seção (uma tabela de características elétricas tem de 1000 a 2000 caracteres). Escolhemos `chunk_size=1000`, que cabe uma tabela pequena ou metade de uma grande, e `chunk_overlap=200`, umas quatro ou cinco linhas de tabela, para que o fim de uma tabela leve junto as linhas de antes. O `add_start_index` guarda a posição do trecho na página, usada para os ids.

In [9]:
def cortar(documentos, tamanho=1000, sobreposicao=200):
    cortador = RecursiveCharacterTextSplitter(chunk_size=tamanho, chunk_overlap=sobreposicao, add_start_index=True)
    return cortador.split_documents(documentos)


trechos = cortar(limpos)
print(len(trechos), "trechos")

202 trechos


### Análise dos trechos

Quantidade de trechos produzidos, distribuição de tamanhos e alguns exemplos. Deve ser mostrado ao menos um caso em que o corte separou uma informação que deveria ficar junta, e se a sobreposição resolveu.

In [10]:
tamanhos = pd.Series([len(t.page_content) for t in trechos])
print(tamanhos.describe().round(0))
print(pd.Series([t.metadata["documento"] for t in trechos]).value_counts())

count     202.0
mean      787.0
std       246.0
min       119.0
25%       648.0
50%       916.0
75%       969.0
max      1000.0
dtype: float64
esp32-datasheet    166
jlcpcb              23
dht22                9
hc-sr04              4
Name: count, dtype: int64


In [11]:
# A tabela de características DC (p. 52) diz que a corrente de 40 mA por pino cai para cerca de 29 mA
# com muitos pinos ligados, numa nota de rodapé abaixo da tabela.
for t in [t for t in trechos if t.metadata.get("pagina") == 52]:
    texto = t.page_content
    print(f"start_index={t.metadata['start_index']}  tem 'source current': {'source current' in texto}  "
          f"tem a nota dos 29 mA: {'29 mA' in texto}")
    print(texto[:150].replace("\n", " | "), "\n...\n", texto[-150:].replace("\n", " | "), "\n")

start_index=0  tem 'source current': True  tem a nota dos 29 mA: False
5 Electrical Characteristics | 5.3 DC Characteristics (3.3 V, 25 °C) | Table 5-3. DC Characteristics (3.3 V, 25 °C) | Parameter Description Min Typ Max Unit 
...
 45 — kΩ | PU | R Resistance of internal pull-down resistor — 45 — kΩ | PD | Chip reset release voltage (CHIP_PU voltage | V 0.75 × VDD 1 — VDD 1 + 0.3 V | IH_nRST 

start_index=799  tem 'source current': False  tem a nota dos 29 mA: True
R Resistance of internal pull-up resistor — 45 — kΩ | PU | R Resistance of internal pull-down resistor — 45 — kΩ | PD | Chip reset release voltage (CHIP_PU vo 
...
 s. | 5.4 RF Current Consumption in Active Mode | The current consumption measurements are taken with a 3.3 V supply at 25 °C of ambient temperature at the 

start_index=1611  tem 'source current': False  tem a nota dos 29 mA: False
5.4 RF Current Consumption in Active Mode | The current consumption measurements are taken with a 3.3 V supply at 25 °C of ambient 

O corte separou a informação da pergunta 6: a linha "High-level source current ... 40 mA" ficou no primeiro trecho da página 52, e a nota de rodapé que diz que a corrente cai para cerca de 29 mA com vários pinos ficou no segundo. A sobreposição de 200 caracteres não resolveu, porque ela repete só o fim do trecho anterior (as linhas dos resistores de pull-up e pull-down), não a linha da corrente, que está no meio da tabela. Para essa tabela, a sobreposição teria que ser do tamanho dela, ou o corte teria que respeitar a tabela inteira.

## Indexação

Armazenamento dos trechos em um vector store, com os embeddings calculados pelo modelo escolhido. Os `ids` devem ser estáveis, para que rodar a célula de novo não duplique os trechos, e o índice deve ser gravado em disco, para não recalcular os embeddings a cada execução.

Usamos o Chroma gravado em `indice/`. O id de cada trecho é um hash do documento, da página (ou seção) e do `start_index`, então rodar a célula de novo substitui os mesmos trechos em vez de duplicar. A coleção leva o tamanho do trecho no nome, para a comparação do fim do notebook usar outra coleção.

In [12]:
def ids_estaveis(trechos):
    chaves = [f"{t.metadata['documento']}|{t.metadata.get('pagina', t.metadata.get('secao'))}|{t.metadata['start_index']}"
              for t in trechos]
    return [hashlib.md5(c.encode()).hexdigest() for c in chaves]


def indexar(trechos, nome):
    vs = Chroma(collection_name=nome, embedding_function=embeddings, persist_directory="indice",
                collection_metadata={"hnsw:space": "cosine"})
    if vs._collection.count() != len(trechos):
        vs.reset_collection()
        vs.add_documents(trechos, ids=ids_estaveis(trechos))
    return vs


vector_store = indexar(trechos, "trechos-1000")
print(vector_store._collection.count(), "trechos no índice")

202 trechos no índice


## Retrieval

Buscas com perguntas do benchmark, mostrando os trechos que voltam e as distâncias, e verificando se o trecho que contém a resposta aparece e em que posição.

O Chroma devolve a distância de cosseno (0 é idêntico, quanto menor, mais perto). A função `acerto` diz em que posição veio o primeiro trecho de cada grupo de fontes da pergunta.

In [13]:
def bate(trecho, fonte):
    m = trecho.metadata
    return m["documento"] == fonte["documento"] and all(m.get(k) == v for k, v in fonte.items() if k != "documento")


def posicoes(resultados, item):
    """Posição (1 = primeiro) do primeiro trecho de cada grupo de fontes, ou None."""
    return [next((i for i, (t, _) in enumerate(resultados, 1) if any(bate(t, f) for f in grupo)), None)
            for grupo in item["fontes"]]


def mostrar(pergunta, k=5, **kwargs):
    resultados = vector_store.similarity_search_with_score(pergunta, k=k, **kwargs)
    for i, (t, d) in enumerate(resultados, 1):
        onde = f"p. {t.metadata['pagina']}" if "pagina" in t.metadata else t.metadata["secao"]
        print(f"{i}. {d:.3f}  {t.metadata['documento']:16} {onde:10} {t.page_content[:90]!r}")
    return resultados


for item in [BENCHMARK[5], BENCHMARK[0], BENCHMARK[11]]:
    print(f"\n[{item['id']}] {item['pergunta']}")
    r = mostrar(item["pergunta"])
    print("posição do trecho certo por grupo:", posicoes(r, item))


[6] Quanta corrente um pino GPIO do ESP32 consegue fornecer?
1. 0.133  esp32-datasheet  p. 45      '4 Functional Description\nPin Assignment\nThe 10 capacitive-sensing GPIOs are listed in Tabl'
2. 0.135  esp32-datasheet  p. 36      '4 Functional Description\n4.8 Digital Peripherals\n4.8.1 General Purpose Input / Output Inte'
3. 0.137  esp32-datasheet  p. 17      '2 Pins\n2.3 IO Pins\n2.3.1 Restrictions for GPIOs and RTC_GPIOs\nAll IO pins of the ESP32 hav'
4. 0.137  esp32-datasheet  p. 37      'configuration.\nFeature List\n• Standard mode (100 Kbit/s)\n• Fast mode (400 Kbit/s)\n• Up to '
5. 0.138  esp32-datasheet  p. 61      'Appendix A\nAppendix A –ESP32 Pin Lists\nA.1. Notes on ESP32 Pin Lists\nTable 6-1. Notes on E'
posição do trecho certo por grupo: [None]

[1] Qual é a faixa de tensão de alimentação do sensor DHT22?
1. 0.158  dht22            p. 1       'Standard AM2302/DHT22 AM2302/DHT22 with big case and wires\nDigital relative humidity & tem'
2. 0.178  esp32-datasheet  p. 4  

### O valor de k e filtros

Comparação de pelo menos dois valores de `k` e, se os metadados permitirem, de uma busca com filtro. A configuração escolhida para a geração deve ser justificada.

Comparamos `k` = 3, 5 e 8 no benchmark inteiro (sem as perguntas sem resposta), contando uma pergunta como acertada quando todos os grupos de fontes aparecem nos `k` trechos. Depois, uma busca com filtro no documento certo, que é o que o nosso agente faz quando sabe de qual componente é a pergunta.

In [14]:
com_fonte = [b for b in BENCHMARK if b["fontes"]]


def taxa_retrieval(vs, k, filtro=False):
    acertos = []
    for item in com_fonte:
        documentos = {f["documento"] for g in item["fontes"] for f in g}
        if filtro and len(documentos) == 1:
            r = vs.similarity_search_with_score(item["pergunta"], k=k, filter={"documento": documentos.pop()})
        else:
            r = vs.similarity_search_with_score(item["pergunta"], k=k)
        acertos.append(all(pos is not None for pos in posicoes(r, item)))
    return sum(acertos) / len(acertos)


pd.DataFrame([{"k": k, "sem filtro": taxa_retrieval(vector_store, k), "com filtro de documento": taxa_retrieval(vector_store, k, True)}
              for k in (3, 5, 8)]).round(2)

,k,sem filtro,com filtro de documento
0,3,0.54,0.62
1,5,0.54,0.62
2,8,0.54,0.62


O `k` não mudou nada: com 3, 5 ou 8 trechos a taxa é a mesma, porque quando a busca acha a fonte ela vem em 1º lugar, e quando não acha ela também não aparece até o 8º. O filtro por documento ganha uma pergunta (de 0,54 para 0,62), mas não serve para a geração, que não sabe o documento de antemão e precisa de vários documentos nas perguntas espalhadas.

Ficamos com `k=5` sem filtro na geração: a função de resposta recebe só a pergunta e não sabe o documento, e as perguntas espalhadas precisam de trechos de dois ou três documentos ao mesmo tempo, o que o filtro impede. Cinco trechos de 1000 caracteres ainda cabem com folga no prompt. A tabela acima mostra quanto se ganha indo para 8.

## Geração

Prompt com uma mensagem de sistema e uma mensagem humana que receba os trechos recuperados e a pergunta, e uma função que, dada uma pergunta, recupere os trechos e devolva a resposta. A resposta deve citar a fonte de onde veio, e o modelo deve dizer que não encontrou quando os trechos não respondem, em vez de inventar.

A mensagem de sistema restringe a resposta aos trechos, pede a citação no formato `[título, p. N]` e manda dizer que não encontrou quando os trechos não respondem. Cada trecho entra no prompt com o título do documento e a página ou seção, para o modelo poder citar.

In [15]:
K = 5

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Você responde perguntas de eletrônica usando só os trechos de datasheets e manuais fornecidos. "
     "Cite a fonte de cada informação logo depois dela, no formato [título, p. N] ou [título, seção]. "
     "Se os trechos não trazem a resposta, responda exatamente 'Não encontrei essa informação nos documentos.' "
     "e não cite fonte. Não use conhecimento de fora dos trechos. Responda em português, em poucas frases."),
    ("human", "Trechos:\n\n{contexto}\n\nPergunta: {pergunta}"),
])
cadeia = prompt | model | StrOutputParser()


def formatar(resultados):
    blocos = []
    for t, _ in resultados:
        onde = f"p. {t.metadata['pagina']}" if "pagina" in t.metadata else t.metadata["secao"]
        blocos.append(f"[{t.metadata['titulo']}, {onde}]\n{t.page_content}")
    return "\n\n---\n\n".join(blocos)


def busca_densa(vs, k=K):
    return lambda pergunta: vs.similarity_search_with_score(pergunta, k=k)


def responder(pergunta, busca=None):
    resultados = (busca or busca_densa(vector_store))(pergunta)
    return {"resposta": cadeia.invoke({"contexto": formatar(resultados), "pergunta": pergunta}), "resultados": resultados}


print(responder(BENCHMARK[5]["pergunta"])["resposta"])

/home/jcassio-dev/www/labs-unidade2/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Não encontrei essa informação nos documentos.


### Teste com perguntas sem resposta

Execução da função com a pergunta do benchmark que os documentos não respondem e com uma pergunta fora do assunto, mostrando o que o modelo respondeu e se citou alguma fonte.

In [16]:
for pergunta in [BENCHMARK[13]["pergunta"], "Qual é a capital da Austrália?"]:
    print(f"Pergunta: {pergunta}\nResposta: {responder(pergunta)['resposta']}\n")

Pergunta: Qual a faixa de temperatura de operação do HC-SR04?
Resposta: Não encontrei essa informação nos documentos.



Pergunta: Qual é a capital da Austrália?
Resposta: Não encontrei essa informação nos documentos.



## Avaliação

A avaliação mede o sistema contra o benchmark, separando as duas etapas que podem falhar: a busca pode não trazer o trecho certo, e o modelo pode errar mesmo com o trecho certo no prompt.

### Benchmark

Registro do benchmark como uma lista de dicionários ou um `DataFrame`, com pelo menos dez perguntas. Cada item deve ter a pergunta, a resposta esperada e a referência da fonte, como a página ou a seção, que permite conferir se a busca acertou.

O benchmark foi escrito antes de testar a busca, a partir da leitura dos documentos. São 14 perguntas: dez fáceis (a resposta está num lugar só), duas com a resposta espalhada em mais de um documento e duas que os documentos não respondem. Cada pergunta tem a resposta esperada e as fontes, como uma lista de grupos: cada grupo é um pedaço da resposta e traz os lugares onde ele aparece (página no PDF, seção no HTML). Uma pergunta espalhada tem um grupo por documento.

In [17]:
pd.DataFrame(BENCHMARK)[["id", "tipo", "pergunta", "resposta", "fontes"]]

,id,tipo,pergunta,resposta,fontes
0,1,fácil,Qual é a faixa de tensão de alimentação do sensor DHT22?,"3,3 a 5,5 V DC (a tabela elétrica dá mínimo 3,3 V, típico 5 V e máximo 6 V).","[[{'documento': 'dht22', 'pagina': 1}, {'documento': 'dht22', 'pagina': 2}, {'documento': 'dht22', 'pagina': 5}]]"
1,2,fácil,De quanto em quanto tempo posso fazer uma leitura do DHT22?,O período de coleta é de 2 segundos: as leituras devem ter pelo menos 2 s de intervalo.,"[[{'documento': 'dht22', 'pagina': 3}, {'documento': 'dht22', 'pagina': 5}]]"
2,3,fácil,"Qual a distância mínima e máxima que o HC-SR04 mede, e com que precisão?","De 2 cm a 400 cm (4 m), com precisão de até 3 mm.","[[{'documento': 'hc-sr04', 'pagina': 1}]]"
3,4,fácil,Qual a tensão de trabalho e a corrente do HC-SR04?,5 V DC e 15 mA.,"[[{'documento': 'hc-sr04', 'pagina': 1}]]"
4,5,fácil,Como converto a largura do pulso do Echo do HC-SR04 em centímetros?,Distância em cm = tempo em µs / 58 (ou tempo do nível alto x 340 m/s / 2).,"[[{'documento': 'hc-sr04', 'pagina': 2}, {'documento': 'hc-sr04', 'pagina': 1}]]"
5,6,fácil,Quanta corrente um pino GPIO do ESP32 consegue fornecer?,"Cerca de 40 mA nos domínios VDD3P3_CPU e VDD3P3_RTC (20 mA no VDD_SDIO), caindo para cerca de 29 mA por pino quando ...","[[{'documento': 'esp32-datasheet', 'pagina': 52}]]"
6,7,fácil,Quanto o ESP32 consome em deep sleep?,10 µA com só o RTC timer e a memória RTC; 150 µA com o coprocessador ULP ligado; 100 µA a 1% de ciclo no padrão moni...,"[[{'documento': 'esp32-datasheet', 'pagina': 30}]]"
7,8,fácil,Quais são os strapping pins do ESP32 e o nível padrão de cada um?,"GPIO0 (pull-up, 1), GPIO2 (pull-down, 0), MTDI (pull-down, 0), MTDO (pull-up, 1) e GPIO5 (pull-up, 1).","[[{'documento': 'esp32-datasheet', 'pagina': 22}]]"
8,9,fácil,Quantos GPIOs programáveis o ESP32 tem e quantos deles são só de entrada?,"34 GPIOs programáveis, dos quais 6 são só de entrada (e 5 são strapping pins).","[[{'documento': 'esp32-datasheet', 'pagina': 4}]]"
9,10,fácil,Qual a tensão máxima absoluta que posso aplicar na alimentação do ESP32?,"3,6 V (a faixa absoluta é de -0,3 V a 3,6 V).","[[{'documento': 'esp32-datasheet', 'pagina': 51}]]"


### Avaliação do retrieval

Para cada pergunta, verificação de que algum dos `k` trechos recuperados vem da fonte esperada, e cálculo da taxa de acerto sobre o benchmark inteiro.

In [18]:
def avaliar_retrieval(busca):
    linhas = []
    for item in BENCHMARK:
        r = busca(item["pergunta"])
        pos = posicoes(r, item)
        linhas.append({"id": item["id"], "tipo": item["tipo"], "posições": pos or "-",
                       "retrieval": all(x is not None for x in pos) if pos else None})
    return pd.DataFrame(linhas)


retrieval = avaliar_retrieval(busca_densa(vector_store))
print("taxa de acerto do retrieval (k=5):", round(retrieval["retrieval"].dropna().mean(), 2))
retrieval

taxa de acerto do retrieval (k=5): 0.54


,id,tipo,posições,retrieval
0,1,fácil,[1],True
1,2,fácil,[None],False
2,3,fácil,[1],True
3,4,fácil,[1],True
4,5,fácil,[1],True
5,6,fácil,[None],False
6,7,fácil,[1],True
7,8,fácil,[1],True
8,9,fácil,[None],False
9,10,fácil,[None],False


### Avaliação das respostas

Comparação de cada resposta gerada com a resposta esperada. O julgamento pode ser feito à mão ou por um modelo como juiz com saída estruturada, que devolva um veredito e uma justificativa. Com o juiz, uma amostra dos vereditos deve ser conferida à mão.

O juiz é o mesmo modelo, com saída estruturada (`Veredito`). Ele recebe a pergunta, a resposta esperada e a resposta gerada, e julga só o conteúdo: a resposta está certa se traz os valores da esperada, sem contradizê-la. Para as perguntas sem resposta, está certa se diz que não encontrou.

In [19]:
class Veredito(BaseModel):
    correta: bool = Field(description="A resposta gerada traz a informação da resposta esperada, sem contradizê-la.")
    justificativa: str = Field(description="Uma frase explicando o veredito.")


juiz = model.with_structured_output(Veredito)
PROMPT_JUIZ = (
    "Você avalia um sistema de perguntas e respostas sobre datasheets.\n"
    "Pergunta: {pergunta}\nResposta esperada: {esperada}\nResposta gerada: {gerada}\n\n"
    "A resposta gerada está correta se traz os valores e fatos principais da resposta esperada sem contradizê-la "
    "(detalhes extras corretos não contam contra). Se a resposta esperada diz que os documentos não informam, "
    "a gerada está correta só se disser que não encontrou a informação, sem inventar um valor."
)


def avaliar_respostas(busca):
    linhas = []
    for item in BENCHMARK:
        r = responder(item["pergunta"], busca)
        v = juiz.invoke(PROMPT_JUIZ.format(pergunta=item["pergunta"], esperada=item["resposta"], gerada=r["resposta"]))
        linhas.append({"id": item["id"], "resposta gerada": r["resposta"], "resposta": v.correta, "justificativa": v.justificativa})
    return pd.DataFrame(linhas)


respostas = avaliar_respostas(busca_densa(vector_store))
print("taxa de acerto das respostas:", round(respostas["resposta"].mean(), 2))
respostas

/home/jcassio-dev/www/labs-unidade2/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


taxa de acerto das respostas: 0.57


,id,resposta gerada,resposta,justificativa
0,1,Não encontrei essa informação nos documentos.,False,"A resposta gerada indica que a informação não foi encontrada, mas o datasheet do DHT22 contém a faixa de tensão de a..."
1,2,Não encontrei essa informação nos documentos.,False,"A resposta gerada afirmou não encontrar a informação, mas o datasheet especifica que o intervalo mínimo entre leitur..."
2,3,"O módulo HC-SR04 possui uma distância mínima de 2cm e uma distância máxima de 4m [HC-SR04 Ultrasonic Ranging Module,...",True,"A resposta gerada apresenta exatamente os valores corretos de distância mínima, máxima e precisão informados na resp..."
3,4,O módulo HC-SR04 possui uma tensão de trabalho de DC 5 V e uma corrente de trabalho de 15 mA [HC-SR04 Ultrasonic Ran...,True,"A resposta gerada informa corretamente a tensão de trabalho de 5 V DC e a corrente de 15 mA, alinhando-se exatamente..."
4,5,Você pode calcular o alcance em centímetros dividindo o tempo do nível alto (em microssegundos) por 58 [HC-SR04 Ultr...,True,"A resposta gerada apresenta corretamente as duas formas de conversão mencionadas na resposta esperada, incluindo a d..."
5,6,Não encontrei essa informação nos documentos.,False,"A resposta gerada indica que não encontrou a informação, mas a resposta esperada continha os dados exatos sobre a co..."
6,7,"O consumo de energia no modo Deep-sleep é de 10 µA [ESP32 Series Datasheet, p. 5]. No padrão monitorado por sensor U...",True,A resposta gerada traz os principais valores de consumo do ESP32 no modo deep sleep que constam na resposta esperada...
7,8,"Os strapping pins do ESP32 e suas configurações padrão são: GPIO0 (Pull-up, valor debit 1), GPIO2 (Pull-down, valor ...",True,"A resposta gerada lista corretamente todos os strapping pins do ESP32 e seus níveis padrão esperados, incluindo a ci..."
8,9,"O ESP32 possui 34 pinos GPIO que podem ser atribuídos a várias funções [ESP32 Series Datasheet, p. 36]. Desses, os p...",True,A resposta gerada menciona corretamente que o ESP32 possui 34 GPIOs programáveis e especifica que 6 deles são exclus...
9,10,Não encontrei essa informação nos documentos.,False,"A resposta gerada indica que a informação não foi encontrada, mas o datasheet do ESP32 especifica claramente que a t..."


Conferência manual dos vereditos do juiz: abaixo vão as respostas inteiras, para lermos cada uma contra a resposta esperada.

In [20]:
for item, (_, linha) in zip(BENCHMARK, respostas.iterrows()):
    print(f"[{item['id']}] juiz: {'certa' if linha['resposta'] else 'errada'}")
    print(f"esperada: {item['resposta']}")
    print(f"gerada:   {linha['resposta gerada']}")
    print(f"juiz:     {linha['justificativa']}\n")

[1] juiz: errada
esperada: 3,3 a 5,5 V DC (a tabela elétrica dá mínimo 3,3 V, típico 5 V e máximo 6 V).
gerada:   Não encontrei essa informação nos documentos.
juiz:     A resposta gerada indica que a informação não foi encontrada, mas o datasheet do DHT22 contém a faixa de tensão de alimentação.

[2] juiz: errada
esperada: O período de coleta é de 2 segundos: as leituras devem ter pelo menos 2 s de intervalo.
gerada:   Não encontrei essa informação nos documentos.
juiz:     A resposta gerada afirmou não encontrar a informação, mas o datasheet especifica que o intervalo mínimo entre leituras é de 2 segundos.

[3] juiz: certa
esperada: De 2 cm a 400 cm (4 m), com precisão de até 3 mm.
gerada:   O módulo HC-SR04 possui uma distância mínima de 2cm e uma distância máxima de 4m [HC-SR04 Ultrasonic Ranging Module, p. 1]. A precisão da medição pode alcançar até 3mm [HC-SR04 Ultrasonic Ranging Module, p. 1].
juiz:     A resposta gerada apresenta exatamente os valores corretos de distância míni

**Conferência manual:** lemos as 14 respostas e concordamos com os 14 vereditos do juiz. Os 6 casos "errada" são todos respostas "Não encontrei essa informação nos documentos" para perguntas que os documentos respondem; os "certa" trazem os valores da resposta esperada. Na pergunta 9 o juiz aceitou "GPIO34 a GPIO39 são só de entrada" como os 6 pinos da resposta esperada, o que está certo.

### Análise

Tabela com a pergunta, o acerto do retrieval e o acerto da resposta, e discussão dos erros mais interessantes, indicando em qual etapa cada um aconteceu. Por fim, um parâmetro deve ser alterado, como o `chunk_size`, o `k` ou o prompt, e o benchmark rodado de novo para comparar os números.

In [21]:
tabela = pd.DataFrame(BENCHMARK)[["id", "tipo", "pergunta"]].merge(retrieval[["id", "posições", "retrieval"]]).merge(
    respostas[["id", "resposta"]])
tabela

,id,tipo,pergunta,posições,retrieval,resposta
0,1,fácil,Qual é a faixa de tensão de alimentação do sensor DHT22?,[1],True,False
1,2,fácil,De quanto em quanto tempo posso fazer uma leitura do DHT22?,[None],False,False
2,3,fácil,"Qual a distância mínima e máxima que o HC-SR04 mede, e com que precisão?",[1],True,True
3,4,fácil,Qual a tensão de trabalho e a corrente do HC-SR04?,[1],True,True
4,5,fácil,Como converto a largura do pulso do Echo do HC-SR04 em centímetros?,[1],True,True
5,6,fácil,Quanta corrente um pino GPIO do ESP32 consegue fornecer?,[None],False,False
6,7,fácil,Quanto o ESP32 consome em deep sleep?,[1],True,True
7,8,fácil,Quais são os strapping pins do ESP32 e o nível padrão de cada um?,[1],True,True
8,9,fácil,Quantos GPIOs programáveis o ESP32 tem e quantos deles são só de entrada?,[None],False,True
9,10,fácil,Qual a tensão máxima absoluta que posso aplicar na alimentação do ESP32?,[None],False,False


In [22]:
# Pergunta 1: o retrieval conta acerto porque veio um trecho da página 1 do DHT22, mas qual trecho?
for t, d in busca_densa(vector_store)(BENCHMARK[0]["pergunta"])[:2]:
    print(f"{d:.3f} {t.metadata}\n{t.page_content}\n")
print([t.page_content[-300:] for t in trechos if t.metadata["documento"] == "dht22" and t.metadata["pagina"] == 1][1:])

0.158 {'pagina': 1, 'titulo': 'AM2302 (DHT22) Datasheet', 'documento': 'dht22', 'source': 'dht22.pdf', 'start_index': 0}
Standard AM2302/DHT22 AM2302/DHT22 with big case and wires
Digital relative humidity & temperature sensor AM2302/DHT22
1. Feature & Application:
*High precision *Outstanding long-term stability
*Capacitive type *Extra components not needed
*Full range temperature compensated *Long transmission distance, up to 100 meters
*Relative humidity and temperature measurement *Low power consumption
*Calibrated digital signal *4 pins packaged and fully interchangeable
2. Description:
AM2302 output calibrated digital signal. It applys exclusive digital-signal-collecting-technique and humidity
sensing technology, assuring its reliability and stability. Its sensing elements is connected with 8-bit single-chip
computer.
Every sensor of this model is temperature compensated and calibrated in accurate calibration chamber and the
calibration-coefficient is saved in type of programme i

### Mudança de parâmetro: `chunk_size` de 1000 para 500

Rodamos o benchmark de novo com trechos de 500 caracteres (sobreposição de 100), numa coleção separada, mantendo `k=5`.

In [23]:
trechos_500 = cortar(limpos, 500, 100)
vs_500 = indexar(trechos_500, "trechos-500")
retrieval_500 = avaliar_retrieval(busca_densa(vs_500))
respostas_500 = avaliar_respostas(busca_densa(vs_500))

comparacao = pd.DataFrame([
    {"chunk_size": 1000, "trechos": len(trechos), "retrieval": retrieval["retrieval"].dropna().mean(), "respostas": respostas["resposta"].mean()},
    {"chunk_size": 500, "trechos": len(trechos_500), "retrieval": retrieval_500["retrieval"].dropna().mean(), "respostas": respostas_500["resposta"].mean()},
]).round(2)
comparacao

/home/jcassio-dev/www/labs-unidade2/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,chunk_size,trechos,retrieval,respostas
0,1000,202,0.54,0.57
1,500,387,0.54,0.50


In [24]:
lado_a_lado = tabela[["id", "tipo", "retrieval", "resposta"]].merge(
    retrieval_500[["id", "retrieval"]].rename(columns={"retrieval": "retrieval 500"})).merge(
    respostas_500[["id", "resposta"]].rename(columns={"resposta": "resposta 500"}))
lado_a_lado[(lado_a_lado["retrieval"] != lado_a_lado["retrieval 500"]) | (lado_a_lado["resposta"] != lado_a_lado["resposta 500"])]

,id,tipo,retrieval,resposta,retrieval 500,resposta 500
3,4,fácil,True,True,True,False
13,14,sem resposta,None,True,None,True


### Segundo experimento: consulta em inglês e busca híbrida

A análise mostrou que a busca densa com a pergunta em português não acha as tabelas do datasheet (perguntas 6, 9 e 10). No nosso Assistente de Bancada, o que resolveu isso foi escrever a consulta em inglês com os termos do datasheet e juntar a busca densa com uma busca por palavras (BM25), intercalando os resultados das duas. Aqui o próprio modelo reescreve a pergunta antes da busca; a pergunta original continua indo para a geração.

In [25]:
from langchain_community.retrievers import BM25Retriever

reescrever = ChatPromptTemplate.from_messages([
    ("system", "Reescreva a pergunta do usuário como uma consulta de busca em inglês, com os termos técnicos que um "
               "datasheet usaria (nomes de tabelas e parâmetros). Responda só com a consulta."),
    ("human", "{pergunta}"),
]) | model | StrOutputParser()

bm25 = BM25Retriever.from_documents(trechos, k=K, preprocess_func=lambda t: re.findall(r"[a-z0-9_]+", t.lower()))


def busca_hibrida(pergunta, k=K):
    consulta = reescrever.invoke({"pergunta": pergunta})
    densos = [t for t, _ in vector_store.similarity_search_with_score(consulta, k=k)]
    vistos, saida = set(), []
    for par in zip(densos, bm25.invoke(consulta)):
        for t in par:
            chave = (t.metadata["documento"], t.metadata.get("pagina", t.metadata.get("secao")), t.metadata["start_index"])
            if chave not in vistos and len(saida) < k:
                vistos.add(chave)
                saida.append((t, None))
    return saida


retrieval_hib = avaliar_retrieval(busca_hibrida)
respostas_hib = avaliar_respostas(busca_hibrida)
pd.concat([comparacao, pd.DataFrame([{"chunk_size": "1000, híbrida em inglês", "trechos": len(trechos),
                                       "retrieval": retrieval_hib["retrieval"].dropna().mean(),
                                       "respostas": respostas_hib["resposta"].mean()}]).round(2)])

/home/jcassio-dev/www/labs-unidade2/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,chunk_size,trechos,retrieval,respostas
0,1000,202,0.54,0.57
1,500,387,0.54,0.50
0,"1000, híbrida em inglês",202,0.77,0.71


In [26]:
tabela[["id", "tipo", "retrieval", "resposta"]].merge(
    retrieval_hib[["id", "posições", "retrieval"]].rename(columns={"posições": "posições híbrida", "retrieval": "retrieval híbrida"})).merge(
    respostas_hib[["id", "resposta", "resposta gerada"]].rename(columns={"resposta": "resposta híbrida"}))

,id,tipo,retrieval,resposta,posições híbrida,retrieval híbrida,resposta híbrida,resposta gerada
0,1,fácil,True,False,[2],True,False,Não encontrei essa informação nos documentos.
1,2,fácil,False,False,[None],False,False,Não encontrei essa informação nos documentos.
2,3,fácil,True,True,[1],True,True,"O módulo HC-SR04 mede uma distância mínima de 2 cm e máxima de 4 metros (400 cm), com precisão de até 3 mm [HC-SR04 ..."
3,4,fácil,True,True,[1],True,True,O módulo HC-SR04 possui uma tensão de trabalho de DC 5 V e uma corrente de trabalho de 15 mA [HC-SR04 Ultrasonic Ran...
4,5,fácil,True,True,[1],True,True,Você pode converter a largura do pulso do Echo em centímetros dividindo o tempo em microssegundos por 58 (`uS / 58 =...
5,6,fácil,False,False,[2],True,True,"Para o domínio de energia VDD3P3_CPU ou VDD3P3_RTC, a corrente de alta nível (high-level source current) típica é de..."
6,7,fácil,True,True,[3],True,True,"O consumo de energia no modo Deep-sleep é de 10 µA [ESP32 Series Datasheet, p. 5, p. 30]. No padrão monitorado por s..."
7,8,fácil,True,True,[1],True,True,"Os strapping pins do ESP32 e seus valores de configuração padrão são: GPIO0 (Pull-up, valor 1), GPIO2 (Pull-down, va..."
8,9,fácil,False,True,[2],True,True,"O ESP32 possui 34 pinos GPIO [ESP32 Series Datasheet, p. 36]. Destes, 6 pinos (GPIOs 34-39) são exclusivos para entr..."
9,10,fácil,False,False,[1],True,True,"A tensão máxima absoluta permitida para a entrada de alimentação (VDDA, VDD3P3, VDD3P3_RTC, VDD3P3_CPU e VDD_SDIO) é..."


### Discussão dos erros

| pergunta | retrieval | resposta | etapa do erro |
|---|---|---|---|
| 1. alimentação do DHT22 | acerto (p. 1) | errada | **retrieval, escondido pela métrica.** Veio o 1º trecho da página 1, sem a tabela; a linha "Power supply 3.3-5.5VDC" está no 2º trecho, que não veio. A métrica por página conta acerto, mas o trecho certo não estava no prompt, e o modelo disse corretamente que não encontrou. |
| 2. intervalo de leitura do DHT22 | erro | errada | **retrieval.** O "Collecting period 2 Second" está numa tabela da p. 5 que a busca densa não trouxe. |
| 6. corrente do GPIO | erro | errada | **retrieval.** A tabela de características DC (p. 52) não aparece para a pergunta em português, como já tínhamos visto no nosso projeto da unidade. |
| 9. GPIOs só de entrada | erro (p. 4) | certa | **benchmark incompleto.** A busca não trouxe a p. 4, mas trouxe a p. 36 e a p. 61, que também dizem que são 34 GPIOs e que o GPIO34 a GPIO39 são só de entrada. A referência do benchmark devia listar essas páginas. |
| 10. tensão máxima absoluta | erro | errada | **retrieval.** Mesma causa da 6, agora na tabela de máximos absolutos (p. 51). |
| 12 e 13. espalhadas | erro parcial | erradas | **retrieval e geração.** A busca trouxe só parte das fontes, e o prompt manda dizer que não encontrou quando os trechos não respondem, então o modelo recusou em vez de responder a parte que tinha. |

O modelo não inventou valores em nenhuma resposta: todos os erros são recusas. O sistema erra por falta de trecho, não por alucinação, o que mostra que a instrução de dizer que não encontrou funcionou (inclusive na pergunta 14 e na pergunta fora do assunto).

### Comparação das mudanças

| configuração | retrieval | respostas |
|---|---|---|
| `chunk_size=1000`, busca densa, pergunta em português | 0,54 | 0,57 |
| `chunk_size=500` | 0,54 | 0,50 |
| `chunk_size=1000`, consulta reescrita em inglês e busca híbrida (densa + BM25) | **0,77** | **0,71** |

Diminuir o trecho não ajudou: a busca acertou as mesmas perguntas, e a pergunta 4 passou a errar na resposta, mesmo com a busca trazendo a página certa (não investigamos se o trecho menor cortou a linha da tabela). O que fez diferença foi a consulta: escrita em inglês, com os termos do datasheet, e combinada com a busca por palavras, ela achou as tabelas das perguntas 6, 9 e 10, e as três respostas passaram a acertar. É a mesma conclusão a que chegamos no Assistente de Bancada, onde o agente é instruído a buscar em inglês e a busca é híbrida.

Continuam errando as perguntas 1 e 2, que dependem de linhas de tabela do DHT22, e as duas espalhadas. Para elas, os próximos passos seriam cortar respeitando as tabelas, fazer uma busca por parte da pergunta (uma por componente) e permitir no prompt uma resposta parcial que diga o que faltou.